# Capstone — mirrors your deployed research paper

This notebook mirrors the structure of our final deployed research paper. We document the research question, data preparation, modeling methodology, comparative results, limitations, and editorial recommendations.

## 1. Question

How can we prioritize existing content items for editorial review and refresh work using search performance and freshness signals to maximize organic search discoverability and prevent traffic decline?

**Decision Supported:** Prioritizing editorial resources to refresh stale or declining content items that still have significant search volume and impressions.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

repo_root = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / "scripts" / "ml_utils.py").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate repository root.")
print(f"Repository root: {repo_root}")

Repository root: c:\Users\kunal\Documents\Project\Internship\FlyRank\Google-Search-Ranking-Discoverability


## 2. Data

We use the anonymized starter dataset `data/raw/content_refresh_anonymized.csv`, which contains 30,000 pseudonymized content items across 32 clients. Each row represents a content item with trailing-90-day search performance and GA4 analytics metrics.

**Exclusions:** We exclude `client_id` and `content_id` as learning features (they are used only as context and for grouping). We also exclude future-window metrics like next-month performance to avoid target leakage.

In [2]:
feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
if feature_path.exists():
    df = pd.read_csv(feature_path)
    print(f"Loaded feature vector containing {len(df):,} rows.")
    print(f"Base decline rate: {df['is_declining_label'].mean():.3f}")
else:
    print("Feature vector file not found. Run preparation scripts first.")

Loaded feature vector containing 30,000 rows.
Base decline rate: 0.542


## 3. Methodology

- **Target Label:** `is_declining_label` (1 if future search trend is down, 0 otherwise).
- **Baseline:** A hand-written transparent scoring rule that combines search visibility (impressions), freshness risk (days since update), search position, and page depth (word count).
- **Validation Split:** Client Holdout (GroupShuffleSplit holding out ~20% of clients) to ensure the model generalizes to unseen sites/clients without leakage.
- **Features:** 18 numeric and 8 categorical features representing search volume, competition level, average position, clicks, impressions, engagement rate, scroll rate, and content freshness.

In [3]:
import json
results_path = repo_root / "outputs" / "model_results.json"
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f"Validation split strategy: {results.get('split_strategy')}")
    print(f"Target variable: {results.get('target')}")
else:
    print("Model results JSON not found. Run training scripts first.")

Validation split strategy: client_holdout
Target variable: is_declining_label


## 4. Results (vs baseline)

We evaluate three models (Decision Tree, Logistic Regression, Random Forest) against our baseline rules on the out-of-sample client holdout split.

In [4]:
if results_path.exists():
    models_data = []
    for name, metrics in results["models"].items():
        models_data.append({
            "Model": name,
            "ROC AUC": metrics["roc_auc"],
            "Avg Precision": metrics["average_precision"],
            "Precision@50": metrics["precision_at_50"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"]
        })
    base = results["baseline"]
    models_data.append({
        "Model": "baseline_rules",
        "ROC AUC": base["baseline_roc_auc"],
        "Avg Precision": base["baseline_average_precision"],
        "Precision@50": base["baseline_precision_at_50"],
        "Recall": np.nan,
        "F1": np.nan
    })
    comparison_df = pd.DataFrame(models_data)
    display(comparison_df)
else:
    print("Results not available.")

,Model,ROC AUC,Avg Precision,Precision@50,Recall,F1
0,decision_tree,0.741520,0.575319,0.62,0.716172,0.633885
1,logistic_regression,0.700291,0.521542,0.40,0.566557,0.566245
2,random_forest,0.750030,0.618219,0.74,0.743674,0.639546
3,baseline_rules,0.626892,0.467607,0.24,NaN,NaN


## 5. Limitations

- **Observational Data Only:** The data is cross-sectional and observational; the findings represent associations and cannot prove causal effect sizes (i.e. we cannot claim refreshing *causes* recovery without an A/B test).
- **No Keyword Context:** The model operates entirely on metadata and performance trends; it does not analyze keyword semantics, search intent changes, or domain-level backlink metrics.

In [5]:
if 'df' in locals():
    print("Missing percentages in primary features:")
    print(df[["days_since_last_update", "avg_position", "ctr", "engagement_rate", "scroll_rate"]].isnull().mean())
else:
    print("Dataset not loaded.")

Missing percentages in primary features:
days_since_last_update    0.0
avg_position              0.0
ctr                       0.0
engagement_rate           0.0
scroll_rate               0.0
dtype: float64


## 6. Ranked recommendations

We recommend prioritizing content updates based on the final blended score queue. Actions are mapped to specific reason codes:
- `expand_and_refresh` for thin, high-impression pages.
- `refresh_and_review_ctr` for low-CTR pages in page-one search positions.
- `refresh_and_review_engagement` for visible pages with poor user behavior.
- `refresh` for general high-risk stale pages.

In [7]:
queue_path = repo_root / "outputs" / "refresh_queue.csv"
if not queue_path.exists():
    queue_path = repo_root / "outputs" / "refresh_queue_sample.csv"

if queue_path.exists():
    queue_df = pd.read_csv(queue_path)
    print("Action Recommendations Summary:")
    print(queue_df["suggested_action"].value_counts())
    print("\nTop 5 Scored Recommendations:")
    display(queue_df[["final_rank", "content_id", "final_refresh_score", "suggested_action", "final_reason_codes"]].head(5))
else:
    print("Scored queue not found.")

Action Recommendations Summary:
suggested_action
monitor                          13083
refresh                           8188
refresh_and_review_ctr            6654
refresh_and_review_engagement     1993
expand_and_refresh                  82
Name: count, dtype: int64

Top 5 Scored Recommendations:


,final_rank,content_id,final_refresh_score,suggested_action,final_reason_codes
0,1,content_1f080331fa2b,81.734212,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...
1,2,content_d6570c51c9bd,81.603243,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
2,3,content_6aa43079fb0c,81.544618,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
3,4,content_72e800a9c214,81.169731,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
4,5,content_e04eb9549989,80.957565,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...


## 7. Artifacts the paper embeds

We load and show the top feature importances and generated charts from the output directory.

In [8]:
if 'results' in locals():
    feat_imp = pd.DataFrame(results["best_model"]["feature_importance_top"])
    print("Best Model Feature Importances:")
    display(feat_imp.head(8))
else:
    print("Feature importance data not found.")

Best Model Feature Importances:


,feature,importance
0,days_with_impressions,0.158144
1,log_impressions_90d,0.128638
2,avg_position,0.109164
3,content_age_days,0.095168
4,char_count,0.042608
5,word_count,0.039609
6,log_clicks_90d,0.034463
7,ctr,0.033295


## 8. 5-Minute Demo Outline

This outline serves as a guide for presenting the findings at the Week-8 showcase:

- **Minute 1: The Question:** Editorial teams waste critical writer hours manually auditing legacy pages, often updating content that does not need it while missing pages in active organic search decline. How do we target the right content refreshes?
- **Minute 2: The Method:** We trained a Random Forest model on 30,000 rows of historical search/engagement data using a client-holdout split to prevent client-level data leakage and ensure real portfolio-independent generalization.
- **Minute 3: The Chart:** Show the Feature Importance chart. Visibility (`days_with_impressions`) and average search position are the primary drivers of decline, interacting non-linearly with page age.
- **Minute 4: The Result:** Our Random Forest model achieved a precision@50 of 0.740, representing a 3.1x performance increase (lift) over the traditional heuristic rules (0.240).
- **Minute 5: The Recommendation:** Deploy the prioritized queue with explicit action labels (`refresh_and_review_ctr` for low-CTR page-one items, `expand_and_refresh` for thin content).

## 9. Shareable Cuts

### Short Social Post (Methodology-focused)
> How do you know when a legacy web page is actually worth refreshing? Static rules like 'update after 180 days' fail because they ignore search performance signals. I built a Random Forest classifier in Python using GroupShuffleSplit validation to predict organic search decline. By blending model probabilities with search visibility opportunity, we improved page targeting precision by 3.1x over traditional editorial heuristics. #MachineLearning #SEO #Python #Productivity

### 3-Sentence Employer-Facing Summary
> I built an organic search discoverability scoring and ranking model in Python to prioritize content updates for editorial teams.
> I trained and validated a Random Forest classifier on a dataset of 30,000 pages, using a client-holdout split to ensure portfolio-independent generalization.
> The system achieved a precision@50 of 74%, providing a 3.1x performance increase over static rules and delivering prioritized, action-labeled queues directly to content strategists.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.